In [27]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/iris/Iris.csv
/kaggle/input/iris/database.sqlite
/kaggle/input/processed-data-credit-score/Score.csv


In [28]:
# Bagging on IRIS (Kaggle) 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Kaggle Iris CSV
df = pd.read_csv("/kaggle/input/iris/Iris.csv")
print(df.head(5))
# Drop ID column
df = df.drop("Id", axis=1)

# Split into features & labels
X = df.drop("Species", axis=1)
y = df["Species"]

# Encode target
y = y.astype('category').cat.codes

# Train/Val/Test: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.80, random_state=42)

# Bagging model
bag_model = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),
    n_estimators=50,
    random_state=42
)

bag_model.fit(X_train, y_train)

print("Validation Accuracy:", accuracy_score(y_val, bag_model.predict(X_val)))
print("Test Accuracy:", accuracy_score(y_test, bag_model.predict(X_test)))

print("\nClassification Report (Test):")
print(classification_report(y_test, bag_model.predict(X_test)))


   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa
Validation Accuracy: 1.0
Test Accuracy: 1.0

Classification Report (Test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00        11

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


In [29]:
# DBSCAN (Kaggle Iris)
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Load dataset
df = pd.read_csv("/kaggle/input/iris/Iris.csv").drop("Id", axis=1)

X = df.drop("Species", axis=1)
y_true = df["Species"].astype('category').cat.codes

# Standardize
X_scaled = StandardScaler().fit_transform(X)

# DBSCAN clustering
db = DBSCAN(eps=0.6, min_samples=5)
labels = db.fit_predict(X_scaled)

# Mask for non-noise points
mask = labels != -1

print("Silhouette Score:", silhouette_score(X_scaled[mask], labels[mask]))
print("Adjusted Rand Index (vs. true labels):", adjusted_rand_score(y_true, labels))
print("Cluster labels found:", set(labels))


Silhouette Score: 0.6405002985705055
Adjusted Rand Index (vs. true labels): 0.4706267335681117
Cluster labels found: {0, 1, -1}


In [30]:
# AdaBoost (Kaggle Credit Score)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("/kaggle/input/processed-data-credit-score/Score.csv")  # <-- change this name if needed


for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop("Credit_Score", axis=1)
y = df["Credit_Score"]

# 80/10/10 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# AdaBoost classifier
ada = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.8,
    random_state=42
)

ada.fit(X_train, y_train)

print("Validation Accuracy :", accuracy_score(y_val, ada.predict(X_val)))
print("Test Accuracy       :", accuracy_score(y_test, ada.predict(X_test)))

print("\nClassification Report (Test):")
print(classification_report(y_test, ada.predict(X_test)))




Validation Accuracy : 0.6682673069227691
Test Accuracy       : 0.6692677070828331

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.56      0.61      0.58      1813
           1       0.68      0.57      0.62      2856
           2       0.70      0.74      0.72      5327

    accuracy                           0.67      9996
   macro avg       0.65      0.64      0.64      9996
weighted avg       0.67      0.67      0.67      9996

